In [ ]:
!pip install scikit-learn seaborn

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [ ]:
test_dataset = datasets.ImageFolder(
    "/content/drive/MyDrive/chest_xray/test",
    transform=transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

class_names = test_dataset.classes
print(class_names)

In [ ]:
model = models.mobilenet_v2(pretrained=False)

model.classifier[1] = nn.Linear(model.last_channel, 2)

model.load_state_dict(torch.load("/content/drive/MyDrive/mobilenet_pneumonia.pth", map_location=device))

model = model.to(device)
model.eval()

In [ ]:
y_true = []
y_pred = []

with torch.no_grad():
    
    for images, labels in test_loader:
        
        images = images.to(device)
        labels = labels.to(device)
        
        outputs = model(images)
        
        _, predicted = torch.max(outputs, 1)
        
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predicted.cpu().numpy())

In [ ]:
accuracy = accuracy_score(y_true, y_pred)

print("Test Accuracy:", accuracy)

In [ ]:
print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

print(cm)

In [ ]:
plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, thresholds = precision_recall_curve(y_true, y_pred)

In [ ]:
plt.figure(figsize=(6,5))

plt.plot(recall, precision)

plt.xlabel("Recall")
plt.ylabel("Precision")

plt.title("Precision–Recall Curve")

plt.show()